In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from IPython.display import Image, display, clear_output
from PIL import Image as PIL_Image
import rasterio

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

# Loading Images

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# create symbolic link to dataset for easier access and shorter filepaths
!ln -s "/content/drive/My Drive/OpenEarthMap_wo_xBD" "/content/LC"

In [ ]:
# read in file paths
with open("LC/train.txt") as f:
  train_paths = [path.strip() for path in f.readlines()]

with open("LC/test.txt") as f:
  test_paths = [path.strip() for path in f.readlines()]

with open("LC/val.txt") as f:
  val_paths = [path.strip() for path in f.readlines()]

ct_tr, ct_te, ct_val = len(train_paths), len(test_paths), len(val_paths)
print(train_paths[0:5], "\n train paths:", ct_tr)
print(test_paths[0:5], "\n test paths:", ct_te)
print(val_paths[0:5], "\n val paths:", ct_val)

In [ ]:
# class for images, uses paths present in _paths lists to extract path values
class img:
  def __init__(self, path):
    self.path = path
    self.raw = "/content/LC/" + path.split('_')[0] + "/images/" + path
    self.label = "/content/LC/" + path.split('_')[0] + "/labels/" + path


In [ ]:
# Identify missing paths in dataset
path_dict = {"train set": train_paths, "test set": test_paths, "validation set": val_paths}
All_Missing_Paths = {"Pairs" : [], "Single Images" : [], "Single Lables" : []}

# Iterate over both the dataset name and the list of paths
for dataset_name, path_list_ref in path_dict.items():
  missing_pairs = []
  missing_single_images = []
  missing_single_labels = []

  # Iterate over the list of paths for each split
  for path in path_list_ref:
    image_path_exists = os.path.exists(img(path).raw)
    label_path_exists = os.path.exists(img(path).label)

    # List of paths without a corresponding raw image and label mask
    if not image_path_exists and not label_path_exists:
      missing_pairs.append(path)

    # List of paths without a raw image
    if not image_path_exists:
      missing_single_images.append(path)

    # List of paths without a label
    if not label_path_exists:
      missing_single_labels.append(path)

  print(dataset_name) # Added print to clarify which set is being processed
  print(
      "example missing pairs:", missing_pairs[:3],
      '\n\tmissing pairs:', len(missing_pairs),
        '\n\tmissing single images:', len(missing_single_images),
        '\n\tmissing single labels:', len(missing_single_labels),
        '\n-----------------------------------------------------------------------------------------')

  # Add Missing path lists to their respective master path list
  All_Missing_Paths["Pairs"].extend(missing_pairs)
  All_Missing_Paths["Single Images"].extend(missing_single_images)
  All_Missing_Paths["Single Lables"].extend(missing_single_labels)

  # Combine all missing paths and get unique ones using a set
  all_paths_to_remove = set(missing_pairs + missing_single_images + missing_single_labels)

  # Filter out the missing paths from the current list (referenced by path_list_ref)
  path_list_ref[:] = [path for path in path_list_ref if path not in all_paths_to_remove]


print("Remaining training pairs:", len(train_paths), '/', ct_tr)
print("Remaining testing pairs:", len(test_paths), '/', ct_te)
print("Remaining validation pairs:", len(val_paths), '/', ct_val)

In [ ]:
# generate list of existing single image paths
existing_single_images = list(
    set(All_Missing_Paths["Single Lables"]) - set(All_Missing_Paths["Single Images"]))
print(existing_single_images[0:5])

## Loading images into a tf.data dataset

In [ ]:
def generate_dataset(path_list_param):
  raws = [img(path).raw for path in path_list_param]
  labels = [img(path).label for path in path_list_param]

  # creates dataset of file paths to the raw image and it's ground truth mask, set as a tuple
  dataset = tf.data.Dataset.from_tensor_slices((raws,labels))
  return dataset

train_dataset = generate_dataset(train_paths)
test_dataset = generate_dataset(val_paths) # since test_paths is empty

# shuffle dataset now to simplyify processing later
train_dataset = train_dataset.shuffle(len(train_dataset), seed = 42)
test_dataset = test_dataset.shuffle(len(test_dataset), seed = 42)

# view dataset head
list(train_dataset.as_numpy_iterator())[:5]

In [ ]:
# function to load in tif files as numpy arrays
def read_tif_to_numpy(path):
    with rasterio.open(path) as src:
        arr = src.read()  # (C, H, W)

    # Convert to (H, W, C)
    arr = np.transpose(arr, (1, 2, 0))
    return arr

def load_tensors(raw_path, label_path, size):
    """
    TensorFlow wrapper that returns:
      raw_tensor:  (H, W, 3), float32 in [0, 1]
      label_tensor: (H, W, 1), int32 in [0, NUM_CLASSES-1]

    Earlier in the model development process, many images failed to load to numpy array of the correct shape, crashing the model after 4 or 5 Epochs.
    These issues have been mostly resolved, but the occasional image will fail to load correctly,
    so the dummy image and label fallback is used to prevent the model can continue from brief hiccups.
    """
    def _load_and_process(raw_p, label_p):
        raw_p_str = raw_p.numpy().decode("utf-8")
        label_p_str = label_p.numpy().decode("utf-8")

        # Dummy fallbacks (already resized and correctly shaped)
        dummy_raw = np.zeros((size[0], size[1], 3), dtype=np.float32)
        dummy_label = np.zeros((size[0], size[1], 1), dtype=np.int32)

        try:
            # ---- Read TIFFs into NumPy ----
            raw_arr = read_tif_to_numpy(raw_p_str)
            label_arr = read_tif_to_numpy(label_p_str)

            # Basic sanity checks
            if raw_arr.ndim != 3 or raw_arr.shape[2] != 3:
                print(f"Warning: raw image {raw_p_str} has shape {raw_arr.shape}, expected (H, W, 3). Returning dummy.")
                return dummy_raw, dummy_label

            if label_arr.ndim != 3 or label_arr.shape[2] != 1:
                print(f"Warning: label {label_p_str} has shape {label_arr.shape}, expected (H, W, 1). Returning dummy.")
                return dummy_raw, dummy_label

            # ---- Resize using TensorFlow, but convert back to NumPy ----
            raw_arr_resized = tf.image.resize(
                raw_arr,
                (size[0], size[1]), # Pass only height and width
                method=tf.image.ResizeMethod.LANCZOS3
            ).numpy().astype(np.float32)

            label_arr_resized = tf.image.resize(
                label_arr,
                (size[0], size[1]), # Pass only height and width
                method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
            ).numpy()

            # Ensure label_arr_resized has the exact (H, W, 1) shape before further processing
            # This step explicitly handles cases where tf.image.resize might output inconsistent ranks
            label_arr_resized = label_arr_resized.reshape((size[0], size[1], 1))

            # ---- Normalize raw image ----
            raw_arr_resized = raw_arr_resized / 255.0

            # ---- Clean and validate labels ----
            # Round + cast + clip to [0, NUM_CLASSES - 1]
            label_arr_resized = np.rint(label_arr_resized).astype(np.int32)
            label_arr_resized = np.clip(label_arr_resized, 0, CLASSES - 1)

            flat = label_arr_resized.reshape(-1)
            if flat.size == 0:
                print(f"Warning: label {label_p_str} is empty after processing. Returning dummy.")
                return dummy_raw, dummy_label

            u = np.unique(flat)
            u_min, u_max = int(u.min()), int(u.max())
            # print("dtype:", label_arr_resized.dtype) # for debugging
            # print("Actual min/max:", u_min, u_max)

            if u_min < 0 or u_max >= CLASSES:
                print(
                    f"Warning: Label values for {label_p_str} are outside [0, {CLASSES-1}]. "
                    f"Values: {u}. Returning dummy."
                )
                return dummy_raw, dummy_label

            # ---- Return as tensors ----
            return (
                tf.convert_to_tensor(raw_arr_resized, dtype=tf.float32),
                tf.convert_to_tensor(label_arr_resized, dtype=tf.int32),
            )

        except Exception as e:
            print(f"Error loading or processing {raw_p_str} / {label_p_str}: {e}. Returning dummy.")
            return dummy_raw, dummy_label

    # Wrap Python function with tf.py_function
    raw_tensor, label_tensor = tf.py_function(
        func=_load_and_process,
        inp=[raw_path, label_path],
        Tout=[tf.float32, tf.int32],
    )

    # Static shapes (important for batching / model)
    raw_tensor.set_shape([size[0], size[1], 3])
    label_tensor.set_shape([size[0], size[1], 1])

    return raw_tensor, label_tensor

# Modeling

## Preparation

In [ ]:
VALIDATION_SIZE = .1
TRAIN_LENGTH_NUM_SAMPLES = len(train_paths)
BATCH_SIZE = 8 # Reduced batch size to prevent OOM error
BUFFER_SIZE = 64 # Datasets where shuffled earlier, so a large buffer is not needed
STEPS_PER_EPOCH = int(TRAIN_LENGTH_NUM_SAMPLES // BATCH_SIZE)
IMG_SIZE = (512,512,3) # Reduced image size to prevent OOM error
CLASSES = 9

### Setup input pipeline

In [ ]:
# Load data set of tensors from file paths for training
train_dataset = train_dataset.map(lambda raw_path, label_path: load_tensors(raw_path, label_path, size=IMG_SIZE), num_parallel_calls=tf.data.AUTOTUNE)

# Load data set of tensors from file paths for validation (using test_dataset from generate_dataset(val_paths))
test_dataset = test_dataset.map(lambda raw_path, label_path: load_tensors(raw_path, label_path, size=IMG_SIZE), num_parallel_calls=tf.data.AUTOTUNE)

# display shape of first tensor pair
for raw, label in list(train_dataset.take(1).as_numpy_iterator()):
  print("raw shape:", raw.shape, "\nlabel shape:", label.shape)

In [ ]:
# Create augmentation layers once outside the map function
augment_raws_layer = tf.keras.layers.RandomFlip(mode="horizontal", seed=42)
augment_labels_layer = tf.keras.layers.RandomFlip(mode="horizontal", seed=42)

# Function to apply augmentations
def apply_augment_ops(raws, labels):
  # Use the pre-instantiated augmentation layers
  raws = augment_raws_layer(raws)
  labels = augment_labels_layer(labels)
  # Add other augmentations here if needed
  return raws, labels

# Input Pipeline
train_batches = (
    train_dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .repeat()
    .prefetch(buffer_size=tf.data.AUTOTUNE))

test_batches = test_dataset.batch(BATCH_SIZE)

In [ ]:
# display examples of raw images and their mask
def display(display_list):
  plt.figure(figsize=(15, 15))

  title = ['Input Image', 'True Mask', 'Predicted Mask']

  for i in range(len(display_list)):
    plt.subplot(1, len(display_list), i+1)
    plt.title(title[i])
    plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
    plt.axis('off')
  plt.show()

# example display of two images
for images, masks in train_batches.take(2):
  sample_image, sample_mask = images[0], masks[0]
  display([sample_image, sample_mask])

### DeepLabV3+

In [ ]:
"""
------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Basic Building Blocks
------------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
# Convolutional block (convolution, normalization, and activation)
def ConvBlock(input,
    num_filters=256,
    dilation_rate=1,
    kernel_size=3,
    use_bias=False,):

  conv = layers.Conv2D(num_filters,
                        kernel_size = kernel_size,
                        dilation_rate = dilation_rate, # 1 = normal convolution, higher is how dspp expands it's field of vision
                        padding = "same",
                        kernel_initializer = "he_normal",
                        use_bias = use_bias)(input)

  batchnorm = layers.BatchNormalization()(conv)
  activation = layers.ReLU()(batchnorm)

  return activation

# Dilated Spatial Pyramid Pooling block
def DilatedSpatialPyramidPooling(dspp_input):
    dims = dspp_input.shape
    pool = layers.AveragePooling2D(pool_size=(dims[-3], dims[-2]))(dspp_input)
    conv = ConvBlock(pool, kernel_size=1, use_bias=True)
    out_pool = layers.UpSampling2D(
        size=(dims[-3] // conv.shape[1], dims[-2] // conv.shape[2]),
        interpolation="bilinear",
    )(conv)

    # Dilated Convolutions at different sizes (rates) are concantenated to capture spatial context at different rates
    out_1 = ConvBlock(dspp_input, kernel_size=1, dilation_rate=1)
    out_6 = ConvBlock(dspp_input, kernel_size=3, dilation_rate=6)
    out_12 = ConvBlock(dspp_input, kernel_size=3, dilation_rate=12)
    out_18 = ConvBlock(dspp_input, kernel_size=3, dilation_rate=18)

    x = layers.Concatenate(axis=-1)([out_pool, out_1, out_6, out_12, out_18])
    output = ConvBlock(x, kernel_size=1)
    return output

In [ ]:
"""
------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Model Architecture
------------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
def DeeplabV3Plus(num_classes, image_size = (512, 512, 3)):
    # Back bone: pretrained resnet50 model
    input = keras.Input(shape=image_size)
    resnet50 = keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_tensor=input
    )
    resnet_output = resnet50.get_layer("conv4_block6_2_relu").output
    DSPP_output = DilatedSpatialPyramidPooling(resnet_output)

    input_a = layers.UpSampling2D(
        size=(image_size[0] // 4 // DSPP_output.shape[1], image_size[1] // 4 // DSPP_output.shape[2]),
        interpolation="bilinear",
    )(DSPP_output)
    input_b = resnet50.get_layer("conv2_block3_2_relu").output

    input_b = ConvBlock(input_b, num_filters=48, kernel_size=1)

    x = layers.Concatenate(axis=-1)([input_a, input_b])
    x = ConvBlock(x)
    x = ConvBlock(x)
    x = layers.UpSampling2D(
        size=(image_size[0] // x.shape[1], image_size[1] // x.shape[2]),
        interpolation="bilinear",
    )(x)
    model_output = layers.Conv2D(num_classes, kernel_size=(1, 1), padding="same")(x)
    return keras.Model(inputs=input, outputs=model_output)


model = DeeplabV3Plus(image_size=IMG_SIZE, num_classes=CLASSES)
model.summary()

In [ ]:
# Plot of the many layers (there's a lot more with this one)
tf.keras.utils.plot_model(model, "model.png", show_shapes=False, show_dtype=False,
                          show_layer_names=True, rankdir='TB', expand_nested=False, dpi=96)

## Training

In [ ]:
# Display model predictions after each epoch
from matplotlib import image
class DisplayInLoop(tf.keras.callbacks.Callback):
  def __init__(self, dataset):
    self.dataset = dataset

  def on_epoch_end(self, epoch, logs=None):
    images, masks = next(iter(self.dataset))

    sample_image, sample_mask = images[0], masks[0]

    pred_mask = self.model.predict(
    tf.expand_dims(sample_image, 0),
    verbose=0)[0]

    pred_mask_class_indices = tf.argmax(pred_mask, axis=-1)
    pred_mask_display = tf.expand_dims(pred_mask_class_indices, axis=-1)

    display([sample_image, sample_mask, pred_mask_display])

In [ ]:
def sparse_categorical_focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        # Ensure y_true is shape (B, H, W)
        y_true = tf.squeeze(y_true, axis=-1)  # only removes the last dim if it's 1

        # Convert to int
        y_true = tf.cast(y_true, tf.int32)

        # Number of classes
        num_classes = tf.shape(y_pred)[-1]

        # One-hot encode
        y_true_one_hot = tf.one_hot(y_true, depth=num_classes)  # (B,H,W,C)

        # Softmax over logits
        y_pred_soft = tf.nn.softmax(y_pred, axis=-1)

        # Compute CE term
        cross_entropy = -tf.math.log(y_pred_soft + 1e-7)

        # Compute focal weights
        weights = alpha * tf.pow(1 - y_pred_soft, gamma)

        # Apply focal weighting ONLY to true class
        focal_loss = y_true_one_hot * weights * cross_entropy

        # Mean over all pixels
        return tf.reduce_mean(focal_loss)
    return loss


In [ ]:
# A MeanIoU metric that works with in training loop
class MeanIoU(tf.keras.metrics.MeanIoU):
    def update_state(self, y_true, y_pred, sample_weight=None):
        # Reshape y_true if it has a channel dimension of 1 (e.g., (None, H, W, 1) -> (None, H, W))
        if y_true.shape[-1] == 1:
            y_true = tf.squeeze(y_true, axis=-1)

        # y_pred comes as logits from the model, apply argmax to get class indices
        y_pred = tf.argmax(y_pred, axis=-1)

        return super().update_state(y_true, y_pred, sample_weight)

# Dice coefficient function for the same
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """
    Multi-class Dice coefficient for sparse labels.
    y_true: (B, H, W) or (B, H, W, 1) with integer class labels
    y_pred: (B, H, W, C) logits
    """
    # Explicitly reshape y_true to remove the last dimension (1)
    # We expect y_true to be (Batch, Height, Width, 1) from the dataset batching.
    # We want it to be (Batch, Height, Width) for tf.one_hot.
    y_true = tf.reshape(y_true, tf.shape(y_true)[:-1]) # Reshape to (B, H, W)

    # Convert logits to probabilities
    y_pred = tf.nn.softmax(y_pred, axis=-1)

    # One-hot encode y_true
    num_classes = tf.shape(y_pred)[-1]
    y_true = tf.one_hot(tf.cast(y_true, tf.int32), num_classes)

    # Compute Dice
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2])
    union = tf.reduce_sum(y_true + y_pred, axis=[1,2])

    dice = (2.0 * intersection + smooth) / (union + smooth)
    return tf.reduce_mean(dice)


In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics = [MeanIoU(num_classes=CLASSES),
                         dice_coefficient,
                         tf.keras.metrics.SparseCategoricalAccuracy()]
)

callbacks = [
    keras.callbacks.ModelCheckpoint("/content/drive/MyDrive/Data 5610 Advanced Machine Learning/Projects/Final_Project/DeepLabv3+_Sat_Segementation.h5", save_best_only=True),
    DisplayInLoop(train_batches)
]

epochs = 25 # Up this if needed

In [ ]:
# Get a single batch from the training data for a dummy predictions
for images, masks in train_batches.take(1):
  dummy_input_image = images[0] # Take the first image from the batch
  break # We only need one

# Perform a dummy prediction to build the model's prediction graph
# This ensures all variables for prediction are created before model.fit starts
_ = model.predict(tf.expand_dims(dummy_input_image, 0), verbose=0)

In [ ]:
# Train the model, doing validation at the end of each epoch.
history = model.fit(train_batches, epochs=epochs, steps_per_epoch=STEPS_PER_EPOCH, validation_data=test_batches, callbacks=callbacks)

## Performance and Prediction

In [ ]:
# graph Loss Metrics
def loss_graph(metric = 'loss'):
  plt.figure(figsize=(10, 5))
  plt.plot(history.history[metric], label='Train ' + metric, color='darkslateblue')
  plt.plot(history.history['val_' + metric], label='Val ' + metric, color='seagreen')
  plt.xlabel('Epoch')
  plt.ylabel(metric)
  plt.title('Training vs. Validation ' + metric)
  plt.legend()
  plt.grid(True)
loss_graph('loss')
loss_graph('dice_coefficient')
loss_graph('mean_io_u_1')
loss_graph('sparse_categorical_accuracy')
plt.show()

In [ ]:
# Create prediction masks for select images
def pred_mask(img_path, size = (512,512)):
  # convert raw img to a tensor
  raw_img_arr = read_tif_to_numpy(img(img_path).raw)

  # resize img with lanzcos3
  img_arr = tf.image.resize(
                raw_img_arr,
                (size[0], size[1]), # Pass only height and width
                method=tf.image.ResizeMethod.LANCZOS3
            ).numpy().astype(np.float32)

  # Normalize input to [0, 1] range to match training data
  img_arr = img_arr / 255.0

  # Predict
  pred = model.predict(img_arr[None, ...])[0]   # remove batch â†’ (512, 512, 9)
  pred_mask_arr = np.argmax(pred, axis=-1)      # (512, 512)

  # plot
  plt.figure(figsize=(15, 4))

  plt.subplot(1, 1, 1)
  plt.title(img_path[:-4])
  plt.imshow(tf.keras.utils.array_to_img(img_arr))
  plt.axis('off')

  plt.subplot(1, 2, 2)
  plt.title("Predicted Class Mask")
  plt.imshow(pred_mask_arr)
  plt.axis("off")
  plt.show()

In [ ]:
import time
import random

random.shuffle(existing_single_images)
iterations = len(existing_single_images)//3
for i in range(iterations):
  pred_mask(existing_single_images[i]) # start at 1st image
  pred_mask(existing_single_images[i + iterations//3]) # start at image a third of the way through images
  pred_mask(existing_single_images[i + 2 * (iterations//3)]) # start at image two thirds of the way through images
  time.sleep(7)
  clear_output(wait=True)

In [ ]:
# print out specific images for comparison
img_title = input("Provide an image title > ")
pred_mask(img_title + ".tif")